# 11.8 模型有看圖還是猜答案？


如果訓練資料十張圖有九張圓，模型不看圖、一律回答圓，就有90%正確率。這個高分對「它會看圖」的證據很弱。本節先用類別平衡的資料計算這種猜測基準，再說明如何換圖檢驗答案是否跟視覺線索變化。

前置是[11.4答案依圖與問題](https://birdhackor.github.io/tiny-perceptron-vlm/11.4.html)與[11.1敏感不等於正確](https://birdhackor.github.io/tiny-perceptron-vlm/11.1.html)。問題都固定為「什麼形狀」，真值由圖片生成條件給定。問題-only基準指只利用文字問題、不讀取圖的預測方法；它能揭露資料分布本來就允許猜多高。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
truth = ["circle", "square", "circle", "square"]
text_only = ["circle"] * len(truth)
accuracy = sum(a == b for a, b in zip(truth, text_only)) / len(truth)
print("平衡資料的固定猜測", accuracy)
imbalanced = ["circle"] * 9 + ["square"]
print("九比一資料的固定猜測", imbalanced.count("circle") / len(imbalanced))

輸入第一組兩圓兩方，`text_only`產生四個圓預測。`zip`將真值與預測逐項配對，比較True計1、False計0，除4得到0.5。第二組九圓一方，`count`數圓，除10得到0.9。輸出變化來自類別比例，模型規則完全沒變；所以資料不平衡會讓固定猜測顯得很厲害。

實際視覺模型先用原圖評正確率，再將同問題配不同圖：圓換方時，正確答案必須從圓變方。應按新圖真值重新評分，不是只看文字變了。如果新回答從圓變「紅」，雖然有變化，仍沒有完成形狀問題。會看圖的模型在換圖後仍可能全部答對，因為我們也按新圖改了正確答案；這正是期待的結果。

另做一個遮蔽檢查：把原圖的視覺內容遮住，問題與參考答案仍保留原題的版本。例如原題答案是circle，遮蔽後仍以circle作參考。這次刻意移除作答所需的視覺線索，觀察原題成績是否下降，叫消融（ablation）。換圖並重標答案是在檢查「是否答對新圖」；遮蔽但保留原答案是在檢查「原題的表現是否依賴視覺線索」。兩種檢查的評分規則要分開記錄。

下圖把問題固定在「什麼形狀」，左邊紅圓換成右邊藍方，箭頭表示只替換圖片資料。正確答案要從circle變square；即使顏色也變了，題目仍只問形狀，不能拿「藍」當成功回答。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_swap_image.svg")))

若遮蔽圖後成績幾乎不變，可能模型沒有利用圖，也可能測試本來能靠問題或分布猜答案；需要回查資料與具體回答，不能單憑一個分數下結論。這段程式只算固定猜測基準，換圖與遮蔽的模型檢查尚未執行。保留未訓練屬性組合，例如訓練紅圓與藍方、測試紅方與藍圓，更能檢驗它是否把顏色當成形狀捷徑。

替圖時應改變目標相關屬性。只把紅圓換成藍圓，形狀答案本來就該保持circle，不能因答案沒變說模型忽略圖。若問顏色，這個替換才應改變回答。先寫出每個幹預下的理想變化，再執行模型，能避免在看到結果後隨意解釋。圖中同時換顏色與形狀，問題明確只選形狀，讓核對仍有唯一目標。

練習把第一組改成九圓一方，先預測一律猜圓得0.9，再執行核對。然後在紙上為其中一張圓圖改成方圖，寫出理想模型應改變的那一個答案。這個對照不需要新術語，卻比只說「回答有變化」更明確地規定了什麼叫有用到圖片。
